# 04 — Energy Model & Candidate Feasibility

## Mục tiêu
Chuẩn hóa feasibility cho tổ hợp `Incident × Ambulance × Hospital` theo:
- availability;
- hospital capability;
- modeled receiving capacity;
- response SLA;
- EV SOC/energy reserve.

## Công thức lõi
\[
E^{required}_{aih}=
(E_{a\to i}+E_{i\to h}+E_{equipment}+E_{h\to charger})(1+b)
\]

và sau mission:
\[
SOC^{charger}_{aih}\ge SOC^{reserve}_a
\]

Notebook này chủ yếu khóa **data contract + QA**. Feasibility cuối cùng trong rolling DES phải dùng trạng thái xe tại decision epoch, không dùng static `SOC_before` cho mọi policy.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

cwd = Path.cwd().resolve()
ROOT = next((p for p in [cwd, cwd.parent, *cwd.parents] if (p/"data"/"sample").exists()), None)
if ROOT is None: raise FileNotFoundError("Không tìm thấy project root.")

DATA = ROOT/"data"/"sample"
OSM = ROOT/"data"/"osmnx"
ROUTING = ROOT/"data"/"routing"
OUT = ROOT/"data"/"feasibility"
OUT.mkdir(parents=True, exist_ok=True)

inc = pd.read_csv(DATA/"incidents.csv.gz")
fleet = pd.read_csv(DATA/"fleet_initial.csv.gz")
hosp = pd.read_csv(DATA/"hospitals.csv")
cap = pd.read_csv(DATA/"hospital_capacity_timeseries.csv.gz")
chg = pd.read_csv(DATA/"charging_stations.csv")

print("Loaded:", len(inc), "incidents;", len(fleet), "fleet rows")

## 1. Energy parameter contract

Không hard-code một "thông số TP.HCM". Các giá trị phải có provenance.

Bản V2 hiện có `battery_capacity_kwh`, `nominal_consumption_kwh_km`, `reserve_soc_pct` trong `fleet_initial`.

In [ ]:
display(fleet[[
    "battery_capacity_kwh",
    "nominal_consumption_kwh_km",
    "reserve_soc_pct",
    "soc_init_pct"
]].describe())

## 2. Hospital capability matrix

In [ ]:
specialties = sorted({s for x in hosp["supported_specialties"] for s in str(x).split("|")})
capability = []
for _, h in hosp.iterrows():
    hs = set(str(h["supported_specialties"]).split("|"))
    for s in specialties:
        capability.append({
            "hospital_id": h["hospital_id"],
            "required_specialty": s,
            "capability_pass": s in hs and int(h["emergency_receiving_flag"]) == 1,
        })
capability = pd.DataFrame(capability)
capability.to_csv(OUT/"hospital_capability_matrix.csv", index=False)
display(capability.head(12))

## 3. Capacity alignment rule

Hospital capacity được lưu theo 15-minute bins. Tại decision epoch, chọn **bin gần nhất không vượt timestamp hiện tại**.  
Đây vẫn là `modeled_receiving_capacity`, không phải giường thực.

In [ ]:
cap["timestamp_bin"] = pd.to_datetime(cap["timestamp_bin"], utc=True)
inc["timestamp"] = pd.to_datetime(inc["timestamp"], utc=True)

print("Capacity interval examples:")
display(cap.head())

## 4. Candidate feasibility contract

Runtime candidate bắt buộc có:

```text
availability_pass
capability_pass
capacity_pass
SOC_pass
SLA_pass
candidate_feasible_hard
```

`candidate_feasible_hard` nên được tạo từ hard constraints, không phải weighted penalty.

In [ ]:
candidate_contract = pd.DataFrame({
    "field": [
        "availability_pass","capability_pass","capacity_pass",
        "SOC_pass","SLA_pass","candidate_feasible_hard"
    ],
    "meaning": [
        "vehicle available at decision epoch",
        "hospital supports required specialty + emergency receiving",
        "modeled receiving capacity > 0",
        "post-mission + charger safety SOC >= reserve",
        "response time <= priority SLA",
        "AND of hard constraints selected by protocol"
    ]
})
candidate_contract

## 5. Energy-stress gate

QA trước đó cho thấy reserve-SOC sensitivity khá phẳng. Trước B4/B6 final experiment phải kiểm tra:
- low initial SOC scenario;
- battery-size range;
- auxiliary/equipment energy;
- uncertainty buffer.

Nếu SOC không làm thay đổi feasible set, paper khó chứng minh marginal value của EV-state integration.